# Model 3 — 04 Evaluation & Metrics

## Purpose
Evaluate the trained models on a held-out test set and save an auditable `evaluation_metrics.json`.

Accuracy is not sufficient for this imbalanced intrusion dataset, so the report includes precision, recall, F1, confusion matrices and per-class metrics.

In [1]:

from pathlib import Path
import sys, json, os, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "EVIDENT" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "behaviour"
MODEL_DIR = PROJECT_ROOT / "models" / "behavior"
ENV_DIR = MODEL_DIR / "environments"
META_DIR = MODEL_DIR / "metadata"
PROD_DIR = MODEL_DIR / "production"

for d in (ENV_DIR, META_DIR, PROD_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

import pandas as pd
import numpy as np

def clean_col(c):
    return " ".join(str(c).strip().replace("\ufeff","").split())

def norm_label(x):
    return "UNKNOWN" if pd.isna(x) else str(x).strip().upper().replace(" ","_")

def family(x):
    x = norm_label(x)
    if x == "BENIGN": return "BENIGN"
    if "PATATOR" in x or "BRUTE" in x: return "BRUTE_FORCE"
    if "PORTSCAN" in x or "PORT_SCAN" in x: return "PORT_SCAN"
    if "DDOS" in x or x.startswith("DOS"): return "DOS_DDOS"
    if "WEB_ATTACK" in x or "SQL_INJECTION" in x or "XSS" in x: return "WEB_ATTACK"
    if "BOT" in x: return "BOTNET"
    if "INFILTRATION" in x: return "INFILTRATION"
    if "HEARTBLEED" in x: return "HEARTBLEED"
    return "OTHER_ATTACK"

files = sorted(DATA_DIR.glob("*.csv"))
print("EVIDENT root:", PROJECT_ROOT)
print("CIC-IDS2017 files:", len(files))
for p in files:
    print(" -", p.name)
assert files, "No CIC-IDS2017 CSV files found."


EVIDENT root: c:\Users\bimal\OneDrive\Desktop\EVIDENT
CIC-IDS2017 files: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [2]:

import pandas as pd, json, joblib, time
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

cache = PROJECT_ROOT/"data"/"processed"/"behavior"/"cicids2017_model3_training.parquet"
schema = json.loads((META_DIR/"feature_schema.json").read_text())
df = pd.read_parquet(cache)
features = schema["feature_columns"]

X=df[features]; yb=df["is_attack"]; yf=df["attack_family"]
X_train,X_test,yb_train,yb_test,yf_train,yf_test=train_test_split(
    X,yb,yf,test_size=.20,random_state=42,stratify=yb
)

def make_model(n):
    return Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
        ("classifier",HistGradientBoostingClassifier(
            max_iter=n,learning_rate=.08,max_leaf_nodes=31,
            l2_regularization=1.0,random_state=42))
    ])

binary_model=make_model(250)
family_model=make_model(300)
binary_model.fit(X_train,yb_train)
mask=yf_train!="BENIGN"
family_model.fit(X_train[mask],yf_train[mask])

t=time.perf_counter()
yb_pred=binary_model.predict(X_test)
elapsed=(time.perf_counter()-t)*1000/max(len(X_test),1)

p,r,f,_=precision_recall_fscore_support(yb_test,yb_pred,average="binary",zero_division=0)

amask=yf_test!="BENIGN"
yf_pred=family_model.predict(X_test[amask])
family_labels=sorted(yf_test[amask].unique())

metrics={
 "binary":{
   "accuracy":float(accuracy_score(yb_test,yb_pred)),
   "precision":float(p),"recall":float(r),"f1":float(f),
   "confusion_matrix":confusion_matrix(yb_test,yb_pred).tolist(),
   "classification_report":classification_report(yb_test,yb_pred,output_dict=True,zero_division=0)
 },
 "attack_family":{
   "accuracy":float(accuracy_score(yf_test[amask],yf_pred)),
   "confusion_matrix":confusion_matrix(yf_test[amask],yf_pred,labels=family_labels).tolist(),
   "labels":family_labels,
   "classification_report":classification_report(yf_test[amask],yf_pred,output_dict=True,zero_division=0)
 },
 "performance":{
   "test_rows":int(len(X_test)),
   "average_binary_prediction_ms":float(elapsed)
 }
}
(META_DIR/"evaluation_metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")
print(json.dumps({
 "binary":metrics["binary"]["f1"],
 "family_accuracy":metrics["attack_family"]["accuracy"],
 "prediction_ms":metrics["performance"]["average_binary_prediction_ms"]
},indent=2))


{
  "binary": 0.9989587589461059,
  "family_accuracy": 0.9961501951270963,
  "prediction_ms": 0.005107966780733818
}
